# Combined Dataset Analysis

วิเคราะห์ข้อมูลจาก **iris** (150 rows) และ **wheat-seeds** (210 rows) ที่รวมเข้าด้วยกันเป็น 360 rows

Dataset ทั้งสองมี schema ต่างกัน ดังนั้นคอลัมน์ที่ไม่มีจะเติม NaN (structural missing)

## 1. Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load & Inspect

In [ ]:
df = pd.read_csv("combined_dataset.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
df.tail(10)

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Dataset Distribution

In [ ]:
print("Rows per dataset:")
print(df["dataset"].value_counts())
print()
print("Class distribution per dataset:")
for ds in df["dataset"].unique():
    subset = df[df["dataset"] == ds]
    target = "species" if ds == "iris" else "Class"
    print(f"\n  [{ds}] target='{target}':")
    print(subset[target].value_counts().to_string(header=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Dataset distribution
df["dataset"].value_counts().plot.bar(ax=axes[0], color=["#6C9BD2", "#E8927C"], edgecolor="black")
axes[0].set_title("Rows per Dataset")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Target distribution per dataset
for ds, color, ax in zip(["iris", "wheat-seeds"], ["#6C9BD2", "#E8927C"], axes):
    subset = df[df["dataset"] == ds]
    target = "species" if ds == "iris" else "Class"
    counts = subset[target].value_counts().sort_index()
    ax2 = axes[1] if ds == "iris" else axes[1]

iris_counts = df[df["dataset"] == "iris"]["species"].value_counts().sort_index()
wheat_counts = df[df["dataset"] == "wheat-seeds"]["Class"].value_counts().sort_index()

x = np.arange(len(iris_counts))
width = 0.35
axes[1].bar(x - width / 2, iris_counts.values, width, label="iris (species)", color="#6C9BD2", edgecolor="black")

x2 = np.arange(len(wheat_counts))
axes[1].bar(x2 + width / 2, wheat_counts.values, width, label="wheat-seeds (Class)", color="#E8927C", edgecolor="black")

axes[1].set_title("Target Distribution per Dataset")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Missing Values Analysis

NaN ที่เห็นไม่ใช่ missing data จริง แต่เป็น **structural missing** จากการรวมคนละ schema

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values("missing_pct", ascending=False)
print("Columns with missing values (structural NaN):")
missing_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
missing_df["missing_pct"].plot.barh(ax=ax, color="#E8927C", edgecolor="black")
ax.set_xlabel("Missing %")
ax.set_title("Missing Values by Column (Structural NaN)")
ax.axvline(x=50, color="red", linestyle="--", alpha=0.5, label="50% threshold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap="YlOrRd")
plt.title("Missing Value Matrix (yellow = NaN)")
plt.tight_layout()
plt.show()

---
## 5. Iris Dataset Analysis

In [ ]:
iris = df[df["dataset"] == "iris"].drop(columns=["dataset", "Area", "Perimeter", "Compactness",
    "Length of Kernel", "Width of Kernel", "Asymmetry Coefficient",
    "Length of Kernel Groove", "Class"])
print(f"Iris shape: {iris.shape}")
iris.describe()

In [ ]:
iris_features = iris.columns[:-1]

# KDE plot
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.flat, iris_features):
    for species in iris["species"].unique():
        subset = iris[iris["species"] == species]
        ax.hist(subset[feature], alpha=0.5, label=species, bins=15, edgecolor="black")
    ax.set_xlabel(feature + " (cm)")
    ax.set_ylabel("Count")
    ax.legend()
    ax.set_title(f"Distribution of {feature}")

plt.suptitle("Iris Dataset — Feature Distribution by Species", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr = iris[iris_features].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="YlGn", fmt=".2f")
plt.title("Iris — Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.flat, iris_features):
    sns.boxplot(data=iris, x="species", y=feature, palette="pastel", ax=ax)
    ax.set_ylabel(feature + " (cm)")
    ax.set_title(f"{feature} by Species")

plt.suptitle("Iris — Boxplot by Species", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 6. Wheat-Seeds Dataset Analysis

In [ ]:
wheat = df[df["dataset"] == "wheat-seeds"].drop(columns=["dataset", "sepal length", "sepal width",
    "petal length", "petal width", "species"])
print(f"Wheat-seeds shape: {wheat.shape}")
wheat.describe()

In [ ]:
wheat_features = wheat.columns[:-1]  # exclude 'Class'

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, feature in zip(axes.flat, wheat_features):
    for cls in sorted(wheat["Class"].unique()):
        subset = wheat[wheat["Class"] == cls]
        ax.hist(subset[feature], alpha=0.5, label=f"Class {cls}", bins=15, edgecolor="black")
    ax.set_xlabel(feature)
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.set_title(feature, fontsize=9)

plt.suptitle("Wheat-Seeds — Feature Distribution by Class", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
corr_wheat = wheat[wheat_features].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_wheat, annot=True, cmap="YlGn", fmt=".2f")
plt.title("Wheat-Seeds — Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, feature in zip(axes.flat, wheat_features):
    sns.boxplot(data=wheat, x="Class", y=feature, palette="pastel", ax=ax)
    ax.set_title(feature, fontsize=9)

plt.suptitle("Wheat-Seeds — Boxplot by Class", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 7. Cross-Dataset Summary Comparison

In [ ]:
# เปรียบเทียบ summary statistics ของแต่ละ dataset
print("=" * 60)
print("Iris Dataset Summary")
print("=" * 60)
print(f"  Rows: {len(iris)}")
print(f"  Features: {list(iris_features)}")
print(f"  Target: species = {list(iris['species'].unique())}")
print(f"  Numeric range:")
for f in iris_features:
    print(f"    {f:20s}  min={iris[f].min():.1f}  max={iris[f].max():.1f}  mean={iris[f].mean():.2f}")

print()
print("=" * 60)
print("Wheat-Seeds Dataset Summary")
print("=" * 60)
print(f"  Rows: {len(wheat)}")
print(f"  Features: {list(wheat_features)}")
print(f"  Target: Class = {sorted(wheat['Class'].unique())}")
print(f"  Numeric range:")
for f in wheat_features:
    print(f"    {f:30s}  min={wheat[f].min():.2f}  max={wheat[f].max():.2f}  mean={wheat[f].mean():.2f}")

---
## 8. Combined Feature Overview

แสดง histogram ของทุก numeric column ใน combined dataset (split by dataset)

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "Class"]

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flat

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    iris_data = df.loc[df["dataset"] == "iris", col].dropna()
    wheat_data = df.loc[df["dataset"] == "wheat-seeds", col].dropna()

    if len(iris_data) > 0:
        ax.hist(iris_data, alpha=0.5, label="iris", bins=15, color="#6C9BD2", edgecolor="black")
    if len(wheat_data) > 0:
        ax.hist(wheat_data, alpha=0.5, label="wheat-seeds", bins=15, color="#E8927C", edgecolor="black")

    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

# ซ่อน subplot ที่เหลือ
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Combined Dataset — Feature Distribution by Source", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 9. Data Type Summary

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.notnull().sum(),
    "null": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(1),
    "n_unique": df.nunique(),
})
summary

In [ ]:
print("\nDone!")
print(f"Combined dataset: {df.shape[0]} rows x {df.shape[1]} columns")